# Notebook 2: Detailed Analysis

**Telecom Churn Case Study**

This notebook covers:
- Data cleaning pipeline
- High-value customer filtering (70th percentile)
- Churn tagging using business rules
- Bivariate analysis: churn vs non-churn
- Feature engineering (action month features)
- Correlation analysis

**Prerequisite:** Run `01_data_exploration.ipynb` first or ensure `data/raw/telecom_churn_data.csv` is available.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path().resolve().parent))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import (
    HIGH_VALUE_PERCENTILE, MISSING_VALUE_THRESHOLD,
    TARGET_COL, GOOD_MONTHS, REPORTS_DIR
)
from src.data_loader import (
    load_telecom_data, clean_telecom_data, get_missing_value_summary
)
from src.analysis import (
    filter_high_value_customers, tag_churners, engineer_features,
    calculate_churn_rate, compare_churn_groups, get_top_correlated_features
)
from src.visualizations import (
    plot_churn_distribution, plot_feature_by_churn,
    plot_median_comparison, plot_correlation_heatmap
)

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('Setup complete.')

## 1. Load & Clean Data

In [ ]:
# Load raw data
df_raw = load_telecom_data()
print(f'Raw shape: {df_raw.shape}')

# Apply cleaning pipeline
df_clean = clean_telecom_data(df_raw)
print(f'Cleaned shape: {df_clean.shape}')

## 2. High-Value Customer Filtering

Retain only customers whose average monthly recharge (June + July) ≥ 70th percentile.

In [ ]:
df_hv = filter_high_value_customers(df_clean, percentile=HIGH_VALUE_PERCENTILE)

print(f'Total customers:       {len(df_clean):,}')
print(f'High-value customers:  {len(df_hv):,} ({len(df_hv)/len(df_clean)*100:.1f}%)')

# Distribution of average_amt_6_7
fig, ax = plt.subplots(figsize=(10, 4))
threshold = df_hv['average_amt_6_7'].min()
df_clean['average_amt_6_7'].hist(bins=50, ax=ax, alpha=0.7, color='#2196F3', label='All customers')
ax.axvline(threshold, color='red', linestyle='--', linewidth=2, label=f'70th pct = {threshold:.0f}')
ax.set_title('Average Recharge Amount (Month 6+7) — High-Value Threshold', fontsize=12)
ax.set_xlabel('Average Recharge Amount (INR)')
ax.set_ylabel('Count')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Churn Tagging

In [ ]:
df_tagged = tag_churners(df_hv)

churn_rate = calculate_churn_rate(df_tagged)
print(f'Churn rate: {churn_rate:.2f}%')
print(df_tagged[TARGET_COL].value_counts())

In [ ]:
# Visualize churn distribution
fig = plot_churn_distribution(df_tagged, save_path=str(REPORTS_DIR / 'churn_distribution.png'))
plt.show()

## 4. Bivariate Analysis: ARPU by Churn Status

In [ ]:
# ARPU median comparison across months
arpu_cols = ['arpu_6', 'arpu_7', 'arpu_8']
arpu_cols_present = [c for c in arpu_cols if c in df_tagged.columns]

if arpu_cols_present:
    fig = plot_median_comparison(
        df_tagged, arpu_cols_present,
        save_path=str(REPORTS_DIR / 'arpu_by_churn.png')
    )
    plt.show()

In [ ]:
# Statistical significance of ARPU differences
for col in arpu_cols_present:
    stats = compare_churn_groups(df_tagged, col)
    print(f'{col}: churn_median={stats["churn_median"]:.0f}, '
          f'non_churn_median={stats["non_churn_median"]:.0f}, '
          f'p={stats["p_value"]:.4f}, significant={stats["significant"]}')

## 5. Feature Engineering

In [ ]:
df_features = engineer_features(df_tagged)

# Show new features
new_features = ['total_call_min_8', 'total_data_8']
for f in new_features:
    if f in df_features.columns:
        print(f'{f}: mean={df_features[f].mean():.1f}, median={df_features[f].median():.1f}')

In [ ]:
# Compare total call minutes in month 8 by churn status
if 'total_call_min_8' in df_features.columns:
    fig = plot_feature_by_churn(
        df_features, 'total_call_min_8', kind='box',
        save_path=str(REPORTS_DIR / 'call_min_8_by_churn.png')
    )
    plt.show()

    stats = compare_churn_groups(df_features, 'total_call_min_8')
    print(f'total_call_min_8: churn_median={stats["churn_median"]:.0f}, '
          f'non_churn_median={stats["non_churn_median"]:.0f}, '
          f'p={stats["p_value"]:.6f}')

## 6. Correlation Analysis

In [ ]:
# Top features correlated with churn
from src.data_loader import drop_date_and_id_columns

df_model_ready = drop_date_and_id_columns(df_features)

# Drop month-9 columns (churn definition columns — would cause leakage)
month9_cols = [c for c in df_model_ready.columns if c.endswith('_9') and c != TARGET_COL]
df_model_ready = df_model_ready.drop(columns=month9_cols)

top_corr = get_top_correlated_features(df_model_ready, n=15)
print('Top 15 features correlated with churn:')
print(top_corr.to_string())

In [ ]:
# Correlation heatmap
fig = plot_correlation_heatmap(
    df_model_ready, n_features=15,
    save_path=str(REPORTS_DIR / 'correlation_heatmap.png')
)
plt.show()

## Summary

Key findings from this analysis:

1. **~30% of customers** qualify as high-value (≥ 70th percentile recharge spend)
2. **~8–9% churn rate** in the high-value segment
3. **ARPU decline** from months 6/7 to month 8 is the strongest churn predictor
4. **Call minutes in month 8** (`total_call_min_8`) drop dramatically for churners
5. **Data usage in month 8** also declines sharply for churners

➡️ Continue to `03_churn_modeling_insights.ipynb` for model building and evaluation.